# Phase sorting from the clock profile

This notebook gives a clock phase to every side of every embryo. The phase comes from the Her7-Venus profile of the side along the PSM. The sides are then ordered by phase and drawn as a kymograph, in which each row is one side. The other channels are drawn in the same row order.

Two methods give the phase. Both reduce each profile to two numbers and plot every side as a point with those two numbers as its coordinates. The points lie around a ring, and the phase of a side is the angle of its point around the centre of the ring. The methods differ in how the two numbers are chosen. In section 3 PCA chooses them. In section 4 they are measured from a posterior and an anterior region of the profile.

The notebook reads the measurements that `IHC_analysis.ipynb` saved. It does not open any image. The kernel has to start in the `analysis_scripts` folder.

In [ ]:
from pathlib import Path

import lmfit
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
# Figure style. Titles start at the left, and the top and right frame lines are removed.
plt.rcParams.update({"font.size": 12, "axes.titlesize": 13, "axes.titlelocation": "left",
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
# Colours of the posterior region and the anterior region in every figure.
POSTERIOR_COLOR, ANTERIOR_COLOR = "#0072B2", "#D55E00"

## Settings

Every setting of the notebook is in the next cell. After a change, run the notebook again from the top.

In [ ]:
print([f"N_1/triple_ihc_{number:03d}" for number in range(2, 11)])

In [ ]:
# Files

# Folder with the saved measurements. It holds one profiles.parquet for each sample.
MEASUREMENTS = Path("ihc_results/measurements")
# Table of distance ranges to leave out, with the columns of IHC_exclusions.csv. The
# ranges are in µm from the posterior end and apply to every channel. An empty string
# keeps every range.
EXCLUSIONS = ""
# Folder where the table of phases is saved.
OUTPUT = Path("ihc_results/phase_sorting")
# Sides to leave out. A full side name such as "N_1/triple_ihc_003_P1/side0" leaves out
# that one side. A shorter name leaves out every side whose name starts with it. For
# example, "N_1/triple_ihc_003" leaves out every position and side of that sample.
EXCLUDE = ["N_2/triple_ihc_020_P1/side0", "N_2/triple_ihc_020_P1/side1", "N_2/triple_ihc_019_P1/side0", "N_2/triple_ihc_019_P1/side1",'N_1']

# Profiles

# Z method of the saved measurements that is read. Its Z-planes or blocks are pooled
# into one profile for each side.
Z_METHOD = "pooled"
# Channel and compartment of the clock profile that the sides are sorted by.
CLOCK = ("Her7-Venus", "nuclear")
# Channels and compartments drawn in the same row order as the clock in section 7.
OTHERS = [("pMEK", "cytoplasmic"), ("ppERK", "nuclear")]

# Smoothing and scaling of each profile (section 2)

# Width in µm of the rolling mean that smooths each profile. A larger width removes
# more noise and also flattens narrow stripes. 0 turns the smoothing off.
SMOOTH_UM = 15.0
smooth_kind = "mean"  # "mean" or "savgol"
# Percentile of the smoothed profile that is subtracted as its baseline. 0 subtracts
# the minimum of the profile. A larger percentile subtracts more.
BASELINE_PERCENTILE = 10.0
# Number that each profile is divided by after its baseline is subtracted. "sd" is
# the standard deviation of the profile. "max" is its maximum. "none" skips the division.
SCALE = "max"
# Number of points on the relative axis. This axis runs from 0 at the posterior end
# of a band to 1 at its anterior end. More points keep finer detail.
GRID_POINTS = 200
# Spacing in µm between the columns of a kymograph that is aligned at the posterior
# or the anterior end.
STEP_UM = 2.0

# PCA (section 3)

# The two principal components whose scores give the phase, counted from 1.
COMPONENTS = (1,2)

# Two regions (section 4)

# Axis on which the regions and their widths are measured. "relative" uses fractions
# of the band length. "um" uses µm.
REGION_AXIS = "relative"
# Start and end of the posterior region, measured from the posterior end.
POSTERIOR_REGION = (0.0, 0.3)
# Start and end of the anterior region, measured from the anterior end.
ANTERIOR_REGION = (0.0, 0.6)
# Fraction of the maximum of a profile that a point has to reach to count toward the
# number of its region. A larger fraction counts only the brightest part of a region.
# 0 counts every point above the baseline percentile.
THRESHOLD = 0.0
# Number that describes each region. "mean" is the average intensity of the counted
# part. "width" is the length of the counted part. "sum" is the area under the counted
# part. "peak" is the maximum in the region. "peak_position" is its position as a
# fraction of the region length from its posterior edge. "weighted" combines
# WEIGHT_METRIC with the width. "peak_only" gives the phase from the peak position of
# this region alone, read as an angle around a circle. The other region is then not
# used. Only one region can be "peak_only".
POSTERIOR_METRIC = "mean"
ANTERIOR_METRIC = "peak_only"
# Intensity metric that "weighted" uses, which is "mean", "sum" or "peak", and its
# weight. A weight of 1 uses only the intensity metric. A weight of 0 uses only the width.
WEIGHT_METRIC = "mean"
WEIGHT = 0.5

# Ring and phase (sections 3 and 4)

# How the centre of the ring is found. "mean" uses the average of the points.
# "circle" fits a circle to the points. "ellipse" fits an ellipse to them.
CENTRE = "ellipse"
# Fraction of a cycle subtracted from every phase of a method. It moves the point of
# the loop that is called phase 0.
START = {"pca": 0.5, "regions": 0.5}
# True makes the phase of a method run around the ring in the opposite direction.
REVERSE = {"pca": True, "regions": True}

# Display

# Alignment of the sorted kymographs in sections 5 and 7. "posterior" and "anterior"
# line up that end of every band. "relative" stretches every band from 0 to 1.
ALIGNMENT = "relative"
# Method whose row order is used in section 7, which is "pca" or "regions".
SORT_BY = "pca"
# Three sides that are followed through the steps in sections 2 and 4. An empty list
# takes three sides spread through the list of sides.
EXAMPLES = []

In [ ]:
# Words used in the titles and axis labels of the figures.
WHERE = {"nuclear": "in nuclei", "cytoplasmic": "in cytoplasm", "all": "in the whole band"}
CLOCK_LABEL = f"{CLOCK[0]} {WHERE[CLOCK[1]]}"
AXIS_LABEL = {"posterior": "Distance from the posterior end (µm)",
              "anterior": "Distance from the anterior end (µm)",
              "relative": "Position along the band (0 posterior, 1 anterior)"}
# Unit of a profile after the steps of section 2, which depends on SCALE.
SCALED_LABEL = {"sd": "Intensity above baseline (standard deviations)",
                "max": "Intensity above baseline (fraction of the maximum)",
                "none": "Intensity above baseline"}[SCALE]
METHOD_LABEL = {"pca": "PCA phase", "regions": "two region phase"}

## 1. Load the profiles

A side is one band of one embryo. Its name is `replicate/sample/side`. Each sample has a saved table with one row for every position along each band. The value at a position is the summed intensity inside the compartment divided by the summed mask coverage. Both sums run over the Z-planes of the chosen Z method.

The length of a band is used as the PSM length of that side. This is correct only for a band that was traced from the posterior tip of the tail bud to the anterior end of the PSM. The relative axis divides every position by this length. A band that stops short of the anterior end is therefore stretched by the wrong amount.

In [ ]:
def read_sample(path):
    """Returns the profiles of one sample, with one row for each side, channel, compartment and position."""
    saved = pd.read_parquet(path, filters=[("method", "==", Z_METHOD)])
    # Add the sums of all Z-planes at each position, then divide once.
    keys = ["replicate", "sample_id", "side", "channel", "compartment", "position_index"]
    pooled = saved.groupby(keys, as_index=False, observed=True).agg(
        distance_um=("distance_um", "first"), total=("masked_sum", "sum"), weight=("mask_weight", "sum"))
    # A position without any mask coverage has no value.
    pooled["value"] = pooled.total / pooled.weight.where(pooled.weight > 0)
    pooled["name"] = pooled.replicate + "/" + pooled.sample_id + "/" + pooled.side
    return pooled


def profiles_of(channel, compartment):
    """Returns {side name: (distance from the posterior end in µm, intensity)} for one channel and compartment."""
    chosen = table[(table.channel == channel) & (table.compartment == compartment)]
    return {name: (side.distance_um.to_numpy(), side.value.to_numpy())
            for name, side in chosen.sort_values("position_index").groupby("name")}


table = pd.concat([read_sample(path) for path in sorted(MEASUREMENTS.glob("*/*/profiles.parquet"))])
# A side is left out when its name starts with an entry of EXCLUDE. A sample name
# therefore leaves out every position and side of that sample.
table = table[~table.name.str.startswith(tuple(EXCLUDE))]

# Values inside an excluded range are removed. Section 2 fills these gaps from their neighbours.
if EXCLUSIONS:
    for row in pd.read_csv(EXCLUSIONS).itertuples():
        inside = ((table.name == f"{row.replicate}/{row.sample_id}/{row.side}")
                  & table.distance_um.between(row.start_um, row.stop_um))
        table.loc[inside, "value"] = np.nan

clock_raw = profiles_of(*CLOCK)
names = list(clock_raw)

# One row for each side. Sections 3 and 4 add the phases to this table.
sides = pd.DataFrame({"name": names})
sides["replicate"] = sides.name.str.split("/").str[0]
sides["embryo"] = sides.name.str.rsplit("/", n=1).str[0]
sides["side"] = sides.name.str.rsplit("/", n=1).str[1]
sides["length_um"] = [clock_raw[name][0][-1] for name in names]
print(f"{len(sides)} sides from {sides.embryo.nunique()} embryos")
sides.groupby("replicate").length_um.describe().round(0)

### The three alignments

A kymograph is an image of many profiles. Each row is one side. Each column is one position along the band. The colour shows the intensity.

The figure draws the same raw profiles in three ways.

- With the posterior ends lined up, distances are in µm from the posterior end. The anterior ends are ragged.
- With the anterior ends lined up, distances are in µm from the anterior end. The posterior ends are ragged.
- On the relative axis, every band is stretched to the same length.

Grey marks positions that a shorter band does not reach. The posterior end is on the left in every panel. The rows are in file order.

The raw brightness differs several fold between sides. With one colour scale for all rows, the brightest sides would set the scale and most rows would be black. Each row is therefore drawn on its own scale, from its lowest value to its highest value.

In [ ]:
def aligned(profiles, order, alignment):
    """Stacks profiles as the rows of a matrix, in the given order.

    Returns the matrix and the position of each column. A column that a shorter band
    does not reach is NaN in that row.
    """
    if alignment == "relative":
        # Divide each distance by the band length, then read every profile at the same fractions.
        grid = np.linspace(0, 1, GRID_POINTS)
        rows = [np.interp(grid, profiles[name][0] / profiles[name][0][-1], profiles[name][1]) for name in order]
        return np.array(rows), grid
    # The columns run from 0 to the length of the longest band, in steps of STEP_UM.
    grid = np.arange(0, max(profiles[name][0][-1] for name in order) + STEP_UM, STEP_UM)
    rows = []
    for name in order:
        distance, values = profiles[name]
        if alignment == "anterior":
            # Count the distance from the anterior end, and reverse the profile to keep the distances increasing.
            distance, values = distance[-1] - distance[::-1], values[::-1]
        rows.append(np.interp(grid, distance, values, left=np.nan, right=np.nan))
    return np.array(rows), grid


def show_kymograph(ax, profiles, order, alignment, title, phases=None):
    """Draws one kymograph with the posterior end on the left and returns the image.

    phases holds the phase of each row from top to bottom. When it is given, the
    vertical axis is marked in phase. Otherwise the vertical axis has no marks.
    """
    matrix, grid = aligned(profiles, order, alignment)
    # The colour scale ignores the lowest and highest 1% of the values.
    low, high = np.nanpercentile(matrix, [1, 99])
    # Positions that a band does not reach are NaN and are drawn in grey.
    image = ax.imshow(matrix, aspect="auto", interpolation="nearest", vmin=low, vmax=high,
                      cmap=plt.get_cmap("inferno").with_extremes(bad="0.85"),
                      extent=(grid[0], grid[-1], len(order), 0))
    if alignment == "anterior":
        # Distance 0 is the anterior end. Reversing the axis puts the posterior end on the left.
        ax.invert_xaxis()
    if phases is None:
        ax.set_yticks([])
    else:
        # Mark the first row that reaches each quarter of the cycle.
        marks = [0, 0.25, 0.5, 0.75]
        ax.set_yticks(np.searchsorted(phases, marks), marks)
    ax.set(title=title, xlabel=AXIS_LABEL[alignment])
    return image


def row_scaled(profiles):
    """Returns the profiles with each one rescaled to run from 0 at its lowest value to 1 at its highest."""
    return {name: (distance, (values - np.nanmin(values)) / (np.nanmax(values) - np.nanmin(values)))
            for name, (distance, values) in profiles.items()}


fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
titles = {"posterior": "Posterior aligned", "anterior": "Anterior aligned",
          "relative": " Stretched to the same length"}
for ax, alignment in zip(axes, titles):
    image = show_kymograph(ax, row_scaled(clock_raw), names, alignment, titles[alignment])
axes[0].set_ylabel("Sides in file order")
fig.colorbar(image, ax=axes, label=f"{CLOCK_LABEL}\n(0 lowest and 1 highest raw value of each row)")
fig.suptitle("Raw profiles in the three alignments", x=0.01, ha="left", fontsize=15)
plt.show()

## 2. Smooth and scale the profiles

Both methods compare the shapes of the profiles. Each profile goes through four steps before the comparison.

1. Gaps are filled from their neighbours. The profile is then smoothed with a rolling mean of width `SMOOTH_UM`. This removes the noise from single nuclei.
2. The baseline is subtracted. The baseline is the `BASELINE_PERCENTILE` percentile of the smoothed profile.
3. The profile is divided by one number, which `SCALE` sets. Two sides in the same phase can differ in brightness because of the staining or the imaging day. After the division, stripes of the same shape have the same height.
4. The profile is read at `GRID_POINTS` evenly spaced points on the relative axis. Every side then has a value in every column, which PCA needs.

The result of steps 1 to 3 is called the scaled profile in the rest of the notebook. The figure follows three sides through the four steps. Each row of panels is one side.

In [ ]:
def scale_profile(distance, values):
    """Returns one profile after each of steps 1 to 3, as a dictionary of arrays."""
    # Step 1. Fill gaps by linear interpolation, then take a rolling mean over SMOOTH_UM.
    filled = pd.Series(values).interpolate(limit_direction="both")
    window = max(1, round(SMOOTH_UM / (distance[1] - distance[0])))
    if smooth_kind == "mean":
        smoothed = filled.rolling(window, center=True, min_periods=1).mean().to_numpy()
    elif smooth_kind == "savgol":
        smoothed = savgol_filter(filled, window, polyorder=1, mode="nearest")

    # Step 2. Subtract the baseline.
    above = smoothed - np.percentile(smoothed, BASELINE_PERCENTILE)
    # Step 3. Divide by the number that SCALE names.
    divisor = {"none": 1.0, "sd": above.std(), "max": above.max()}[SCALE]
    return {"raw": values, "smoothed": smoothed, "above baseline": above, "scaled": above / divisor}


def scaled(raw):
    """Returns {side name: (distance, scaled profile)} from raw profiles."""
    return {name: (distance, scale_profile(distance, values)["scaled"]) for name, (distance, values) in raw.items()}


clock = scaled(clock_raw)
# Step 4. X has one row for each side and one column for each point of the relative axis.
X, relative = aligned(clock, names, "relative")

# Without EXAMPLES, take the sides one quarter, one half and three quarters of the way through the list.
examples = EXAMPLES or [names[i] for i in np.linspace(0, len(names) - 1, 5).astype(int)[1:4]]

fig, axes = plt.subplots(3, 4, figsize=(18, 8), constrained_layout=True)
for row, name in zip(axes, examples):
    distance, values = clock_raw[name]
    steps = scale_profile(distance, values)
    row[0].plot(distance, steps["raw"], color="0.7", lw=0.8, label="raw")
    row[0].plot(distance, steps["smoothed"], color="k", label="smoothed")
    row[1].plot(distance, steps["above baseline"], color="k")
    row[2].plot(distance, steps["scaled"], color="k")
    row[3].plot(relative, X[names.index(name)], color="k")
    # The legend title names the side that this row of panels shows.
    row[0].legend(title=name, fontsize=7, title_fontsize=8,frameon=False,)
    row[0].set_ylabel("Raw intensity")
    row[1].set_ylabel("Intensity above baseline")
    # The unit goes on a second line, which keeps the labels of neighbouring rows apart.
    row[2].set_ylabel(SCALED_LABEL.replace(" (", "\n("))
    row[3].set_ylabel(SCALED_LABEL.replace(" (", "\n("))
division = {"sd": "3. Division by the standard deviation", "max": "3. Division by the maximum",
            "none": "3. No division"}[SCALE]
for ax, title in zip(axes[0], ["1. Smoothing", "2. Baseline subtraction", division, "4. Relative axis"]):
    ax.set_title(title)
for ax in axes[-1, :3]:
    ax.set_xlabel(AXIS_LABEL["posterior"])
axes[-1, 3].set_xlabel(AXIS_LABEL["relative"])
fig.suptitle("The four preprocessing steps on three sides", x=0.01, ha="left", fontsize=15)
plt.show()

The scaled profiles of all sides on the relative axis are the input of PCA. The rows are still in file order.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5), constrained_layout=True)
image = show_kymograph(ax, clock, names, "relative", "Scaled profiles before sorting")
ax.set_ylabel("Sides in file order")
fig.colorbar(image, ax=ax, label=f"{CLOCK_LABEL}\n{SCALED_LABEL}")
plt.show()

## 3. Phase from PCA

The scaled profiles form a table. Each row is one side. Each column is one position on the relative axis. PCA describes how the rows differ from each other with a small number of patterns.

- The **mean profile** is the average of all rows. PCA subtracts it from every row. What remains is how each side differs from the average side.
- A **principal component** is a pattern along the band. The first component is the pattern that accounts for the most variation between sides. The second component accounts for the most of the variation that the first leaves. Each later component continues in the same way.
- The **loading** of a component at a position is the weight of that position in the pattern.
- The **score** of a side on a component is the amount of that pattern in the side. It is the sum over positions of the loading times the profile with the mean subtracted.
- The **explained variance** of a component is the fraction of the total variation between sides that it accounts for.

PCA has no knowledge of the clock. It finds the largest differences between sides, whatever their cause. A wave that moves along the band gives two components with similar explained variance. Their loadings have the same stripes, shifted along the band. The right panel of the figure shows whether the two chosen components look like this.

In [ ]:
mean_profile = X.mean(axis=0)
# The singular value decomposition of the mean-subtracted table gives the components.
U, S, Vt = np.linalg.svd(X - mean_profile, full_matrices=False)
loadings = Vt                       # one row for each component, one column for each position
scores = U * S                      # one row for each side, one column for each component
explained = S**2 / np.sum(S**2)     # fraction of the variance that each component explains
first, second = COMPONENTS[0] - 1, COMPONENTS[1] - 1

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), constrained_layout=True)
axes[0].plot(relative, mean_profile, color="k")
axes[0].set(title="Mean profile of all sides", xlabel=AXIS_LABEL["relative"], ylabel=f"{CLOCK_LABEL}\n{SCALED_LABEL}")

# The first ten components in grey, with the two chosen components drawn over them in black.
axes[1].bar(np.arange(1, 11), explained[:10], color="0.8", label="other components")
axes[1].bar(COMPONENTS, explained[[first, second]], color="k", label="chosen components")
axes[1].set(title=f"The chosen components explain {explained[first] + explained[second]:.0%} of the variance",
            xlabel="Principal component", ylabel="Fraction of the variance explained", xticks=np.arange(1, 11))
axes[1].legend()

axes[2].plot(relative, loadings[first], color="k", lw=2, label=f"component {first + 1}")
axes[2].plot(relative, loadings[second], color="0.6", lw=2, label=f"component {second + 1}")
axes[2].set(title="Pattern of each chosen component", xlabel=AXIS_LABEL["relative"],
            ylabel="Loading (weight of each position)")
axes[2].legend()
plt.show()

## converting to a phase


Each side is now a point. Its score on one component is x. Its score on the other component is y. Over one clock cycle, x rises and falls once, and y does the same. When x and y are out of step, the point travels once around a ring. The position of a side on the ring is its phase.

Before the points are drawn, each of the two numbers is rescaled across the sides. Its mean is subtracted and the result is divided by its standard deviation. The unit of both axes is then one standard deviation, whatever the units of the two numbers were.

The phase is the angle of a point around the centre of the ring. It is written as a fraction of a cycle from 0 to 1. Phase 1 is the same as phase 0. `CENTRE` sets how the centre is found.

- "mean" uses the average of the points. This is the centre of the ring only if the sides are spread evenly around the cycle.
- "circle" and "ellipse" fit an outline to the points with `lmfit`. The fit minimises the distance from each point to the outline, measured along the line through the centre. This fit does not need an even spread of sides. For an ellipse, the angle is measured after the ellipse is stretched back to a circle.

The fit keeps the centre inside the range of the points. It keeps each radius between 0.2 and 4 standard deviations. Without these limits, a fit to points that do not form a ring can return a very large outline. Only a nearly straight piece of such an outline passes through the points.

Three statements hold for the phase from either method.

- The phase orders the sides around the cycle. Equal steps in phase are not equal steps in time.
- The point called phase 0 and the direction of the phase are arbitrary. `START` and `REVERSE` set them.
- The phase has a meaning only if the points form a ring with an empty centre. Points that fill a disc have no reliable angle.

In [ ]:
def rescale(values):
    """Subtracts the mean of a set of numbers and divides by their standard deviation."""
    return (values - values.mean()) / values.std()


def around_ring(ring, x, y):
    """Returns each point moved and stretched to make the fitted ring a circle of radius 1 around 0."""
    # Move the centre to 0, turn the ellipse upright, then divide by its two radii.
    dx, dy = x - ring["x0"], y - ring["y0"]
    cos, sin = np.cos(ring["tilt"]), np.sin(ring["tilt"])
    return (dx * cos + dy * sin) / ring["a"], (-dx * sin + dy * cos) / ring["b"]


def ring_point(ring, turn, radius=1.0):
    """Returns the x and y of the point at a fraction of a turn around the ring. A radius of 1 is on the outline."""
    # This is the reverse of around_ring.
    angle = 2 * np.pi * np.asarray(turn)
    u, v = radius * ring["a"] * np.cos(angle), radius * ring["b"] * np.sin(angle)
    cos, sin = np.cos(ring["tilt"]), np.sin(ring["tilt"])
    return ring["x0"] + u * cos - v * sin, ring["y0"] + u * sin + v * cos


def outline_gap(x, y, x0, y0, a, b, tilt):
    """Returns the distance from each point to the outline, along the line from the centre through the point.

    x and y are the coordinates of the points. x0 and y0 are the centre of the outline.
    a and b are its two radii. tilt is the angle between the a radius and the x axis.
    The result is positive for a point outside the outline and negative for a point inside it.
    """
    # How far each point is from the centre.
    from_centre = np.hypot(x - x0, y - y0)
    # The direction of each point seen from the centre, as an angle from the x axis.
    direction = np.arctan2(y - y0, x - x0)
    # The same direction, now measured from the a radius instead of from the x axis.
    from_a = direction - tilt
    # How far the outline is from the centre in that direction. This is the equation of
    # an ellipse written with the centre as the origin: a point at distance r and angle
    # from_a lies on the ellipse when (r cos(from_a) / a)^2 + (r sin(from_a) / b)^2 = 1.
    # Solving this for r gives the line below. For a circle, a = b and the line gives a.
    outline = a * b / np.hypot(b * np.cos(from_a), a * np.sin(from_a))
    # The gap between the point and the outline, both measured along the same direction.
    return from_centre - outline


# An lmfit model built from outline_gap. lmfit reads the arguments of outline_gap:
# x and y are the data it is given, and x0, y0, a, b and tilt are the numbers it may change.
ring_model = lmfit.Model(outline_gap, independent_vars=["x", "y"])


def fit_ring(x, y):
    """Fits the outline that CENTRE names to the points and returns the full lmfit result.

    The fitted numbers are in result.best_values, and result.fit_report() prints the
    quality of the fit and the uncertainty of each number.
    """
    # The numbers that describe the outline, each with the value lmfit starts from and
    # the limits it must stay within.
    ring = ring_model.make_params(
        # The centre starts at (0, 0), which is the average of the rescaled points, and
        # stays inside the range of the points.
        x0=dict(value=0, min=x.min(), max=x.max()),
        y0=dict(value=0, min=y.min(), max=y.max()),
        # Rescaled points that are spread evenly on a circle lie at a radius of sqrt(2).
        a=dict(value=np.sqrt(2), min=0.2, max=4),
        b=dict(value=np.sqrt(2), min=0.2, max=4),
        tilt=dict(value=0, min=-np.pi / 2, max=np.pi / 2, vary=True),
    )
    if CENTRE == "mean":
        # Nothing is fitted. Every number is held at its starting value, so the centre
        # stays at the average of the points.
        for name in ring:
            ring[name].set(vary=False)
    if CENTRE == "circle":
        ring["b"].set(expr="a")          # a circle has two equal radii
    if CENTRE == "ellipse":
        ring["tilt"].set(vary=True)      # an ellipse can also be tilted
    # A point on the outline has a gap of 0. The fit therefore compares the gaps with 0
    # and changes the numbers until the sum of the squared gaps is as small as it can be.
    return ring_model.fit(np.zeros_like(x), ring, x=x, y=y)


def phase_of(ring, x, y, method):
    """Returns the phase of each point as a fraction of a cycle, from 0 to 1."""
    u, v = around_ring(ring, x, y)
    turn = np.arctan2(v, u) / (2 * np.pi)
    direction = -1 if REVERSE[method] else 1
    return (direction * turn - START[method]) % 1


def show_ring(ax, x, y, ring, phase, method, xlabel, ylabel):
    """Draws the sides as points coloured by phase, with the centre, the outline and the phase 0 line."""
    ax.plot(*ring_point(ring, np.linspace(0, 1, 200)), color="0.6", lw=1, label="outline")
    ax.plot(ring["x0"], ring["y0"], "+", color="k", markersize=14, label="centre")
    # Phase 0 lies at START turns from the x direction, counted in the direction that REVERSE sets.
    direction = -1 if REVERSE[method] else 1
    ax.plot(*ring_point(ring, direction * START[method], radius=np.array([0, 1.6])), color="k", lw=1.5, label="phase 0")
    for replicate, marker in zip(sorted(sides.replicate.unique()), "o^sD"):
        chosen = (sides.replicate == replicate).to_numpy()
        points = ax.scatter(x[chosen], y[chosen], c=phase[chosen], cmap="twilight", vmin=0, vmax=1,
                            marker=marker, s=70, edgecolor="k", linewidth=0.5)
        # An empty grey point gives the legend the marker of the replicate without a phase colour.
        ax.scatter([], [], marker=marker, color="0.7", edgecolor="k", linewidth=0.5, s=70,
                   label=f"replicate {replicate} (sides)")
    ax.set(xlabel=xlabel, ylabel=ylabel, aspect="equal", title="Phase is the angle around the centre")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncols=3)
    ax.figure.colorbar(points, ax=ax, label="Phase (fraction of a cycle)", ticks=[0, 0.25, 0.5, 0.75, 1])

The figure shows the sides in PC space. Each point is one side. The marker shape shows its replicate. The phase increases from the phase 0 line around the centre.

In [ ]:
pca_x, pca_y = rescale(scores[:, first]), rescale(scores[:, second])
# The full lmfit result of the fit, and the fitted numbers taken from it.
pca_ring_results = fit_ring(pca_x, pca_y)
pca_ring = pca_ring_results.best_values
pca_ring_results.fit_report()
sides["phase_pca"] = phase_of(pca_ring, pca_x, pca_y, "pca")

fig, ax = plt.subplots(figsize=(10,6), constrained_layout=True)
show_ring(ax, pca_x, pca_y, pca_ring, sides.phase_pca.to_numpy(), "pca",
          f"Score on component {first + 1} (standard deviations)",
          f"Score on component {second + 1} (standard deviations)")
plt.show()

## 4. Phase from two regions

This method chooses the two numbers by hand. One number describes the posterior domain of the clock. The other describes the anterior stripe. `POSTERIOR_REGION` is measured from the posterior end. `ANTERIOR_REGION` is measured from the anterior end. `REGION_AXIS` sets the axis of both.

A point of a region counts toward its number when its value is at least `THRESHOLD` times the maximum of its scaled profile. The points that count are called the counted part of the region. Each region is described by one of these numbers.

- **mean** is the average intensity of the counted part.
- **width** is the length of the counted part. In the posterior region this is the length of the posterior domain.
- **sum** is the area under the profile over the counted part. It equals the mean times the width.
- **peak** is the maximum of the profile in the region.
- **peak_position** is the distance from the region's posterior edge to its peak divided by the region length.
- **weighted** is `WEIGHT` times `WEIGHT_METRIC` plus `1 - WEIGHT` times the width. The two have different units. Each is therefore first rescaled across the sides to a mean of 0 and a standard deviation of 1.
- **peak_only** is the same peak position as peak_position, but it gives the phase by itself. The other region is not used. Only one region can be peak_only.

A region without a counted part has a mean, a sum and a width of 0.

The method assumes that each number rises and falls once in a clock cycle. It also assumes that the two numbers are out of step. If they peak together, or if one peaks when the other is lowest, the points fall on a line and the phase cannot be read. The ring figure below shows which case holds.

The next figure shows the two regions on the three sides of section 2. The thick part of each profile is the counted part. Each legend gives the number of each region.

In [ ]:
def region_numbers(distance, values, region, from_anterior):
    """Returns the numbers that describe one region of one scaled profile.

    Also returns which points are inside the region and which of them count.
    """
    x = distance / distance[-1] if REGION_AXIS == "relative" else distance
    if from_anterior:
        x = x[-1] - x
    inside = (x >= region[0]) & (x <= region[1])
    counted = inside & (values >= THRESHOLD * values.max())
    step = abs(x[1] - x[0])
    peak_x = x[inside][values[inside].argmax()]
    peak_position = ((region[1] - peak_x) if from_anterior else (peak_x - region[0])) / (region[1] - region[0])
    # peak_only reads the same peak position. Section 4 turns it into an angle.
    numbers = {"mean": 0.0, "sum": 0.0, "width": 0.0, "peak": values[inside].max(),
               "peak_position": peak_position, "peak_only": peak_position}
    if counted.any():
        numbers.update(mean=values[counted].mean(), sum=values[counted].sum() * step, width=counted.sum() * step)
    return numbers, inside, counted


def region_table(region, from_anterior):
    """Returns one row for each side with every number that describes the region."""
    numbers = pd.DataFrame([region_numbers(*clock[name], region, from_anterior)[0] for name in names])
    numbers["weighted"] = WEIGHT * rescale(numbers[WEIGHT_METRIC]) + (1 - WEIGHT) * rescale(numbers["width"])
    return numbers


def show_region(ax, name, region, from_anterior, color, label):
    """Shades one region of one side and thickens the counted part of its profile."""
    distance, values = clock[name]
    x = distance / distance[-1] if REGION_AXIS == "relative" else distance
    _, inside, counted = region_numbers(distance, values, region, from_anterior)
    ax.axvspan(x[inside].min(), x[inside].max(), color=color, alpha=0.12, lw=0)
    # Points outside the counted part are set to NaN, which leaves them undrawn.
    ax.plot(x, np.where(counted, values, np.nan), color=color, lw=3, label=label)


if POSTERIOR_METRIC == ANTERIOR_METRIC == "peak_only":
    raise ValueError('Only one region can be "peak_only".')

posterior = region_table(POSTERIOR_REGION, False)[POSTERIOR_METRIC].to_numpy()
anterior = region_table(ANTERIOR_REGION, True)[ANTERIOR_METRIC].to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), sharey=True, constrained_layout=True)
for ax, name in zip(axes, examples):
    distance, values = clock[name]
    index = names.index(name)
    ax.plot(distance / distance[-1] if REGION_AXIS == "relative" else distance, values, color="k", lw=0.8)
    # ax.axhline(THRESHOLD * values.max(), color="0.5", ls="--", lw=0.8, label="threshold")
    # When one region is peak_only, the other region is still drawn but its legend says it is not used.
    posterior_label = ("posterior not used" if ANTERIOR_METRIC == "peak_only"
                       else f"posterior {POSTERIOR_METRIC} {posterior[index]:.2f}")
    anterior_label = ("anterior not used" if POSTERIOR_METRIC == "peak_only"
                      else f"anterior {ANTERIOR_METRIC} {anterior[index]:.2f}")
    show_region(ax, name, POSTERIOR_REGION, False, POSTERIOR_COLOR, posterior_label)
    show_region(ax, name, ANTERIOR_REGION, True, ANTERIOR_COLOR, anterior_label)
    ax.set(title=name, xlabel=AXIS_LABEL["relative"] if REGION_AXIS == "relative" else AXIS_LABEL["posterior"])
    # The legend sits below the panel, where it cannot cover the profile.
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncols=3, fontsize=10)
axes[0].set_ylabel(f"{CLOCK_LABEL}\n{SCALED_LABEL}")
fig.suptitle("The two regions on three sides", x=0.01, ha="left", fontsize=15)
plt.show()

Each side is a point. Its posterior number is x. Its anterior number is y. Both are rescaled across the sides as in section 3. The centre, the outline and the phase are also found as in section 3.

When one region is peak_only, its peak position is a fraction of a cycle. It is turned into an angle, and each side is placed at the cosine and the sine of that angle. A peak at 0.98 then lies next to a peak at 0.02. The points lie on the circle of radius 1 around (0, 0) by construction. This circle is used as the ring, nothing is fitted, and `CENTRE` is not used.

In [ ]:
if "peak_only" in (POSTERIOR_METRIC, ANTERIOR_METRIC):
    region, position = ("posterior", posterior) if POSTERIOR_METRIC == "peak_only" else ("anterior", anterior)
    # The peak position runs from 0 to 1 over the region. As an angle, 0 and 1 are the same point.
    angle = 2 * np.pi * position
    regions_x, regions_y = np.cos(angle), np.sin(angle)
    # The points lie on the circle of radius 1 around (0, 0), so this circle is the ring.
    regions_ring = {"x0": 0.0, "y0": 0.0, "a": 1.0, "b": 1.0, "tilt": 0.0}
    xlabel, ylabel = f"Cosine of the {region} peak position", f"Sine of the {region} peak position"
    xcolor = ycolor = POSTERIOR_COLOR if region == "posterior" else ANTERIOR_COLOR
else:
    regions_x, regions_y = rescale(posterior), rescale(anterior)
    # The full lmfit result of the fit, and the fitted numbers taken from it.
    regions_ring_results = fit_ring(regions_x, regions_y)
    regions_ring = regions_ring_results.best_values
    xlabel = f"Posterior region {POSTERIOR_METRIC} (standard deviations)"
    ylabel = f"Anterior region {ANTERIOR_METRIC} (standard deviations)"
    xcolor, ycolor = POSTERIOR_COLOR, ANTERIOR_COLOR
sides["phase_regions"] = phase_of(regions_ring, regions_x, regions_y, "regions")

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
show_ring(ax, regions_x, regions_y, regions_ring, sides.phase_regions.to_numpy(), "regions", xlabel, ylabel)
# The axis labels take the colours of the regions in the figure above.
ax.xaxis.label.set_color(xcolor)
ax.yaxis.label.set_color(ycolor)
plt.show()

## 5. Sorted kymographs

The figure shows the scaled clock profiles with the rows ordered by the phase from each method. Phase 0 is at the top. The marks on the vertical axis show the first row that reaches each quarter of the cycle.

A phase that follows the clock gives stripes that move steadily in one direction from row to row. The phase is a loop. The bottom row therefore continues at the top row.

In [ ]:
# For each method, the row order and the phase of each row from top to bottom.
ranks = {method: np.argsort(sides[f"phase_{method}"].to_numpy()) for method in METHOD_LABEL}
orders = {method: [names[i] for i in ranks[method]] for method in METHOD_LABEL}
sorted_phases = {method: sides[f"phase_{method}"].to_numpy()[ranks[method]] for method in METHOD_LABEL}

fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
for ax, method in zip(axes, METHOD_LABEL):
    image = show_kymograph(ax, clock, orders[method], ALIGNMENT, f"Rows ordered by the {METHOD_LABEL[method]}",
                           sorted_phases[method])
    ax.set_ylabel("Phase (fraction of a cycle)")
fig.colorbar(image, ax=axes, label=f"{CLOCK_LABEL}\n{SCALED_LABEL}")
plt.show()

## 6. Checks

### The two sides of an embryo

Both sides of an embryo are in the same phase. Each side was given its phase separately. A method that works therefore gives the two sides similar phases. Similar phases show that the phase is a property of the embryo. They do not show that this property is the clock. The sorted kymographs in section 5 show that.

Each point in the figure is one embryo. Phase 0 and phase 1 are the same point of the cycle. A side at 0.95 is therefore close to a side at 0.05. To show this, the vertical axis gives the phase of the second side at the equivalent value closest to the first side. This value can lie below 0 or above 1.

The difference between the two sides lies between 0 and 0.5. Phases given at random have a median difference of 0.25.

The figure gives three numbers that measure how close the points lie to the line of equal phase. Pearson's $r$ measures how tightly the points follow a straight line of any slope and offset. The bias factor $C_b$ measures how far that straight line lies from the line of equal phase. $C_b$ equals 1 when both sides have the same mean phase and the same spread of phases. A shift or a difference in spread between the sides lowers $C_b$. The concordance correlation coefficient (CCC) is $r$ multiplied by $C_b$. It equals 1 only when every point lies on the line of equal phase.

The CCC of the phases $x$ and $y$ of the two sides is

$$\mathrm{CCC} = \frac{2 s_{xy}}{s_x^2 + s_y^2 + (\bar{x} - \bar{y})^2}.$$

Here $\bar{x}$ and $\bar{y}$ are the mean phases over the embryos, $s_x^2$ and $s_y^2$ are their variances and $s_{xy}$ is their covariance.

Scatter around the line of equal phase lowers $r$. A shift or a tilt away from that line lowers $C_b$. All three numbers compare the differences between the sides with the spread of phases across embryos. Embryos that cover the whole cycle give higher values than embryos that cluster at one phase when the sides differ by the same amount.

In [ ]:
def nearest_equivalent(reference, phase):
    """Returns each phase at the equivalent value closest to its reference.

    A phase of 0.95 beside a reference of 0.05 is returned as -0.05.
    """
    return reference + (phase - reference + 0.5) % 1 - 0.5


def agreement(x, y):
    """Returns the CCC, Pearson's r and the bias factor C_b of the paired values x and y.

    The CCC measures how close the points (x, y) lie to the line y = x. It is the product of r and C_b.
    """
    # The values become plain arrays so that the variances divide by the number of points, as in Lin's definition.
    x, y = np.asarray(x), np.asarray(y)
    covariance = np.mean((x - x.mean()) * (y - y.mean()))
    ccc = 2 * covariance / (x.var() + y.var() + (x.mean() - y.mean()) ** 2)
    # Pearson's r measures how tightly the points follow their own straight line.
    r = np.corrcoef(x, y)[0, 1]
    # C_b is the part of the CCC that measures how far that straight line lies from y = x.
    return ccc, r, ccc / r


fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
for ax, method in zip(axes, METHOD_LABEL):
    # One row for each embryo, with the phase of each of its sides in a column.
    pairs = sides.pivot(index="embryo", columns="side", values=f"phase_{method}").dropna()
    one, other = pairs.iloc[:, 0], nearest_equivalent(pairs.iloc[:, 0], pairs.iloc[:, 1])
    # The line of equal phase is drawn across the whole axis, because points near phase 0 or 1 can lie outside [0, 1].
    ax.axline((0, 0), slope=1, color="0.6", lw=1, label="equal phase")
    ax.scatter(one, other, color="k", s=30, label="one embryo")
    ccc, r, c_b = agreement(one, other)
    # r and C_b show whether a low CCC comes from scatter or from a shift between the sides.
    ax.text(0.98, 0.02, "\n".join([rf"$r$ = {r:.2f}", rf"$C_b$ = {c_b:.2f}",
                                   f"median difference = {np.median(np.abs(other - one)):.2f}"]),
            transform=ax.transAxes, ha="right", va="bottom", fontsize=12,fontweight="bold",)
    ax.set(title=f"The CCC between the sides is {ccc:.2f}",
           xlabel=f"Phase of one side ({METHOD_LABEL[method]})", ylabel="Phase of the other side",
           xlim=(-0.2,1.2), aspect="equal", xticks=[0, 0.5, 1], yticks=[0, 0.5, 1])
    ax.legend(loc="upper left")
plt.show()

### Agreement between the methods

Each point in the left panel is one side. The two methods agree when the points follow the line of equal phase. The points follow this line only after `START` and `REVERSE` give the two methods the same start and the same direction. The vertical axis gives the two region phase at the equivalent value closest to the PCA phase, as in the figure above. The left panel also gives the CCC, $r$ and $C_b$, with the PCA phase as $x$ and the two region phase as $y$.

### Band length

The other two panels show the band length of each side against its phase. Within one somite stage the PSM lengthens through the clock cycle. The band length then rises once around the loop. This holds only for embryos that are close in stage and for bands that cover the whole PSM.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5), constrained_layout=True)
# The two region phase of each side is given at the equivalent value closest to its PCA phase.
pca, regions = sides.phase_pca, nearest_equivalent(sides.phase_pca, sides.phase_regions)
ccc, r, c_b = agreement(pca, regions)
axes[0].axline((0, 0), slope=1, color="0.6", lw=1, label="equal phase")
axes[0].scatter(pca, regions, color="k", s=30, label="one side")
axes[0].text(0.98, 0.02, "\n".join([rf"$r$ = {r:.2f}", rf"$C_b$ = {c_b:.2f}",
                                    f"median difference = {np.median(np.abs(regions - pca)):.2f}"]),
             transform=axes[0].transAxes, ha="right", va="bottom", fontsize=12)
axes[0].set(title=f"The CCC between the methods is {ccc:.2f}", xlabel="PCA phase (fraction of a cycle)",
            ylabel="Two region phase", xlim=(-0.2,1.2), aspect="equal", xticks=[0, 0.5, 1], yticks=[0, 0.5, 1])
axes[0].legend(loc="upper left")
for ax, method in zip(axes[1:], METHOD_LABEL):
    # The first replicate has filled markers and the later replicates have open markers.
    for replicate, marker in zip(sorted(sides.replicate.unique()), "o^sD"):
        chosen = sides[sides.replicate == replicate]
        ax.scatter(chosen[f"phase_{method}"], chosen.length_um, edgecolor="k",
                   facecolor="k" if marker == "o" else "none", marker=marker, s=30, label=f"replicate {replicate}")
    ax.set(title=f"Band length against the {METHOD_LABEL[method]}", xlabel="Phase (fraction of a cycle)",
           ylabel="Band length (µm)", xlim=(-0.1, 1), xticks=[0, 0.25, 0.5, 0.75, 1])
    ax.legend()
plt.show()

## 7. Other channels in the same order

The figure shows the clock and the channels in `OTHERS`. The rows are in the order of the `SORT_BY` phase. The other channels go through the steps of section 2 as the clock does. With `SCALE` set to "sd" or "max", each row shows the shape of a profile without its overall brightness. Each panel has its own colour scale.

In [ ]:
channels = [CLOCK] + OTHERS
fig, axes = plt.subplots(1, len(channels), figsize=(6 * len(channels), 6), constrained_layout=True)
for ax, (channel, compartment) in zip(axes, channels):
    image = show_kymograph(ax, scaled(profiles_of(channel, compartment)), orders[SORT_BY], ALIGNMENT,
                           f"{channel} {WHERE[compartment]}", sorted_phases[SORT_BY])
    fig.colorbar(image, ax=ax, label=SCALED_LABEL)
axes[0].set_ylabel("Phase (fraction of a cycle)")
fig.suptitle(f"Channels with the rows ordered by the {METHOD_LABEL[SORT_BY]}", x=0.01, ha="left", fontsize=15)
plt.show()

## 8. Save the phases

The table has one row for each side, with its phase from each method. The phase of an embryo is the mean of the phases of its sides. The mean is taken around the loop. The mean of 0.95 and 0.05 is therefore 0.

In [ ]:
def loop_mean(phases):
    """Returns the mean of phases around the loop."""
    # Average the phases as directions on a circle, then turn the average direction back into a phase.
    angle = 2 * np.pi * np.asarray(phases)
    return np.arctan2(np.sin(angle).mean(), np.cos(angle).mean()) / (2 * np.pi) % 1


for method in METHOD_LABEL:
    sides[f"embryo_phase_{method}"] = sides.groupby("embryo")[f"phase_{method}"].transform(loop_mean)
OUTPUT.mkdir(parents=True, exist_ok=True)
sides.to_csv(OUTPUT / "phases.csv", index=False)
print("Saved", OUTPUT / "phases.csv")
sides.head()

In [ ]:
# The saved table has one row for each side. The columns embryo_phase_pca and
# embryo_phase_regions hold the phase of the embryo that the side belongs to.
phase_table = pd.read_csv(OUTPUT / "phases.csv")

# The clock and the other channels, each after the steps of section 2.
channels = [CLOCK] + OTHERS
channel_profiles = {channel: scaled(profiles_of(*channel)) for channel in channels}

for method in METHOD_LABEL:
    # Order the sides by the phase of their embryo. Sorting by embryo name next keeps the
    # two sides of an embryo together when two embryos have the same phase.
    ordered = phase_table.sort_values([f"embryo_phase_{method}", "embryo", "side"])
    order = ordered.name.tolist()
    phases = ordered[f"embryo_phase_{method}"].to_numpy()

    for alignment in ["posterior", "anterior", "relative"]:
        fig, axes = plt.subplots(1, len(channels), figsize=(6 * len(channels), 6), constrained_layout=True)
        for ax, (channel, compartment) in zip(axes, channels):
            image = show_kymograph(ax, channel_profiles[(channel, compartment)], order, alignment,
                                   f"{channel} {WHERE[compartment]}", phases)
            fig.colorbar(image, ax=ax, label=SCALED_LABEL)
        axes[0].set_ylabel("Embryo phase (fraction of a cycle)")
        fig.suptitle(f"Rows ordered by the embryo {METHOD_LABEL[method]}, {titles[alignment].lower()}",
                     x=0.01, ha="left", fontsize=15)
        plt.show()
